In [3]:
import os 
os.environ['SPARK_HOME'] = "/Users/mukesh/opt/spark-3.5.1-bin-hadoop3"
os.environ['JAVA_HOME'] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
# os.environ['PATH'] is usually handled by these two, but setting it explicitly can't hurt:
os.environ['PATH'] = os.environ['SPARK_HOME'] + "/bin:" + os.environ.get('PATH', '')

In [4]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from mysql_spark import ConnectDB

In [5]:
HOST = "127.0.0.1"
USER = "root"
PASSWORD = "Paridhi@2019#"  
DATABASE = "dw_poc"
SOURCE_TABLE = "source_data"
TARGET_TABLE = "processed_results"

db = ConnectDB(host=HOST, password=PASSWORD, user=USER, database=DATABASE)



Initialising the database configuration...


In [7]:
## THIS IS THE WORKAROUND TO LOAD MYSQL CONNECTOR JAR IN SPARK

from pyspark.sql import SparkSession
import os
import glob

# Build a list of likely search roots relative to the notebook process cwd
cwd = os.getcwd()
search_roots = [
    cwd,
    os.path.join(cwd, 'jars'),
    os.path.abspath(os.path.join(cwd, '..')),
    os.path.abspath(os.path.join(cwd, '..', 'jars')),
    os.path.abspath(os.path.join(cwd, '..', '..')),
    os.path.abspath(os.path.join(cwd, '..', '..', 'jars')),
    # explicit project root (helpful if kernel cwd is SCD/)
    os.path.abspath(os.path.join(os.path.expanduser('~'), 'Desktop', 'TEST_AGAIN', 'jars'))
]

patterns = ['*mysql*connector*.jar', 'mysql-connector*.jar', '*mysql*.jar']

matches = []
for root in search_roots:
    for pat in patterns:
        matches.extend(glob.glob(os.path.join(root, '**', pat), recursive=True))

# Also fallback: global recursive search under repo root (limited depth)
repo_root = os.path.abspath(os.path.join(cwd, '..'))
matches.extend(glob.glob(os.path.join(repo_root, '**', '*mysql*connector*.jar'), recursive=True))

matches = sorted(set(matches))
mysql_jar = matches[0] if matches else None

if mysql_jar:
    print(f"Found MySQL JDBC driver jar: {mysql_jar}")
    spark = (
        SparkSession.builder.appName("SCD")
        .config("spark.jars", mysql_jar)
        .config("spark.driver.extraClassPath", mysql_jar)
        .getOrCreate()
    )
    print('Configured Spark with jar on driver and executors.')
else:
    print("No MySQL JDBC driver jar found in expected locations. Looked in:")
    for r in search_roots:
        print(' -', r)
    print('You can place the connector jar in one of those folders (e.g. ./jars) and restart the kernel.')
    spark = SparkSession.builder.appName("SCD").getOrCreate()

print("Spark-Version *** :", spark.version)


Found MySQL JDBC driver jar: /Users/mukesh/Desktop/TEST_AGAIN/jars/mysql-connector-j-9.4.0.jar


25/10/31 17:30:32 WARN Utils: Your hostname, mukeshs-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.29.221 instead (on interface en0)
25/10/31 17:30:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/10/31 17:30:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/31 17:30:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Configured Spark with jar on driver and executors.
Spark-Version *** : 3.5.1


In [8]:
# Create SparkSession

from pyspark.sql import SparkSession

spark = (
        SparkSession.builder.appName("SCD")
        .config("spark.jars", mysql_jar)
        .config("spark.driver.extraClassPath", mysql_jar)
        .getOrCreate()
    )

#### INSERT  DATA  INTO  SOURCE  &  TARGET  FOR  THE  INITIAL  SNAPSHOT 

In [ ]:
# We have out table in mysql , need to insert some records for simulation 
from datetime import date
query1 = "insert into employee_src values(1,'MUKESH','HR',5000,'2025-10-30')"
query2 = "insert into employee_src values(2,'YASH','IT',6000,'2025-10-30')"
query3 = "insert into employee_src values(3,'CHARLIE','FINANCE',8000,'2025-10-30')"

SRC = [query1, query2, query3]
for query in SRC:
    db.execute_query(query)

tar1 = "insert into employee_trg values (1,'Mukesh','HR',5000,date(2025,9,1),date(9999,12,31),True)"
tar2 = "insert into employee_trg values (2,'Bob','IT',6000,date(2025,9,15),date(9999,12,31),True)"
tar3 = "insert into employee_trg values (3,'Charlie','Finance',15000,date(2025,9,10),date(9999,12,31),True)"
           
TRG = [tar1, tar2, tar3]
for query in TRG:
    db.execute_query(query)




Standard Query executed successfully. Rows affected: 1
Standard Query executed successfully. Rows affected: 1
Standard Query executed successfully. Rows affected: 1
Error executing standard query: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near ',9,1),date(9999,12,31),True)' at line 1
Error executing standard query: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near ',9,15),date(9999,12,31),True)' at line 1
Error executing standard query: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near ',9,10),date(9999,12,31),True)' at line 1


In [13]:
# Read data from source table into Spark DataFrame

df_src = db.read_mysql_table_to_spark_df(spark, 'employee_src')

df_src.printSchema()

df_src.show()



--- Spark Read: Reading Entire Table 'employee_src' ---
Successfully read MySQL data into Spark DataFrame. Schema:
root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_date: date (nullable = true)

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_date: date (nullable = true)

+------+-----+----------+------+--------------+
|emp_id| name|department|salary|effective_date|
+------+-----+----------+------+--------------+
|     4|Alice| Marketing|  7000|    2025-10-30|
|     4|Alice| Marketing|  7000|    2025-10-30|
+------+-----+----------+------+--------------+



In [11]:
# READ TARGET TABLE INTO A DATAFRAME 


df_trg = db.read_mysql_table_to_spark_df(spark, 'employee_trg')

df_trg.printSchema()

df_trg.show()


--- Spark Read: Reading Entire Table 'employee_trg' ---
Successfully read MySQL data into Spark DataFrame. Schema:
root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_start_date: date (nullable = true)
 |-- effective_end_date: date (nullable = true)
 |-- flag: boolean (nullable = true)

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_start_date: date (nullable = true)
 |-- effective_end_date: date (nullable = true)
 |-- flag: boolean (nullable = true)

+------+----+----------+------+--------------------+------------------+----+
|emp_id|name|department|salary|effective_start_date|effective_end_date|flag|
+------+----+----------+------+--------------------+------------------+----+
+------+----+----------+------+--------------------+-------